# ADRS vs DRS on the Goldstein et al. QP Benchmark

**Reference**: Goldstein, O'Donoghue, Setzer, Baraniuk — *Fast Alternating Direction Optimization Methods* (2014), Section 7.4.

Their paper runs Fast ADMM with restart on:
$$\min_{x} \;\frac{1}{2}x^\top Q x + q^\top x \quad \text{s.t.} \quad -\mathbf{1} \leq x \leq \mathbf{1}$$
with $n=500$, $\mathrm{cond}(Q) = 10^8$, and reports Figure 9(a): convergence of primal and dual residuals.

**Why this is a good benchmark for us:**
- ADMM on this problem is equivalent to DRS with $g_1 = \frac{1}{2}x^\top Qx + q^\top x$ and $g_2 = \iota_{[-1,1]^n}$
- The ill-conditioning ($10^8$) makes convergence slow — hundreds of iterations — so algorithm differences are clearly visible
- $g_1$ is smooth and strongly convex, $g_2$ is an indicator: **DRS is the natural solver**; APG could handle $g_1$ but not the hard box constraint in this split form
- Goldstein et al. show Fast ADMM+Restart dramatically outperforms vanilla ADMM here: we can check whether our ADRS-M reproduces that gap

**Key proximal operators:**
- $\mathrm{prox}_{\alpha g_1}(z) = (Q + \alpha^{-1}I)^{-1}(\alpha^{-1}z - q)$ — one linear solve (precompute factorisation)
- $\mathrm{prox}_{g_2}(z) = \mathrm{clip}(z, -1, 1)$ — elementwise box projection


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams.update({'font.size': 12, 'lines.linewidth': 2,
                     'axes.grid': True, 'grid.alpha': 0.3})

COLORS = {'DRS': '#1f77b4', 'ADRS': '#ff7f0e',
          'ADRS-M': '#2ca02c', 'ADRS-M-NM': '#d62728'}
STYLES = {'DRS': '-', 'ADRS': '--', 'ADRS-M': '-.', 'ADRS-M-NM': ':'}


## Problem setup

In [ ]:
# ── Problem parameters ────────────────────────────────────────────────
n          = 500
target_cond = 1e8
ALPHA      = 1.0      # DRS stepsize
N_ITER     = 2000     # iterations

# Build Q with condition number 10^8
# Eigenvalues log-spaced from 1 to 10^8
V    = np.linalg.qr(np.random.randn(n, n))[0]
eigs = 10**np.linspace(0, np.log10(target_cond), n)
Q    = V @ np.diag(eigs) @ V.T
Q    = (Q + Q.T) / 2                            # ensure exact symmetry

# Random linear term — scaled to push solution onto boundary
q    = np.random.randn(n) * 100

print(f"n = {n},  cond(Q) = {np.linalg.cond(Q):.2e}")
print(f"Eigenvalue range: [{eigs.min():.2e}, {eigs.max():.2e}]")

# ── Proximal operators ────────────────────────────────────────────────
# prox_{ALPHA * g1}(z) = (Q + (1/ALPHA)*I)^{-1} * ((1/ALPHA)*z - q)
# Precompute the matrix factorisation once
M_factor = Q + (1/ALPHA) * np.eye(n)           # matrix to invert
print("\nFactorising (Q + (1/alpha)*I) ... ", end="")
M_inv = np.linalg.inv(M_factor)                 # ~0.5s for n=500
print("done.")

def prox_quad(z, alpha):
    """prox_{alpha * (0.5 x^T Q x + q^T x)}: quadratic proximal operator."""
    return M_inv @ ((1/alpha) * z - q)

def prox_box(z, alpha):
    """prox_{indicator{-1 <= x <= 1}}: box projection."""
    return np.clip(z, -1, 1)

def objective(x):
    return 0.5 * x @ Q @ x + q @ x

# ── Reference optimum via long DRS run ───────────────────────────────
print("\nComputing F* via 5000-iteration reference DRS ... ", end="")
z = np.zeros(n)
for _ in range(5000):
    x = prox_quad(z, ALPHA)
    u = prox_box(2*x - z, ALPHA)
    z = z + u - x
F_STAR = objective(u)
n_active = int(np.sum(np.abs(u) > 0.999))
print("done.")
print(f"F* = {F_STAR:.4f}")
print(f"Active constraints at optimum: {n_active}/{n}  (Goldstein et al. report ~126)")

z0 = np.zeros(n)   # shared starting point, same as Goldstein et al.


## Algorithm implementations

Same four algorithms as before. The only problem-specific piece is which `prox_quad` / `prox_box` gets called.


In [ ]:
def run_drs(z0, niters):
    z = z0.copy(); resids, objs = [], []
    for _ in range(niters):
        x = prox_quad(z, ALPHA)
        u = prox_box(2*x - z, ALPHA)
        r = np.linalg.norm(u - x)
        z = z + u - x
        resids.append(r); objs.append(objective(u))
    return np.array(resids), np.array(objs)


def run_adrs(z0, niters):
    z, z_prev = z0.copy(), z0.copy()
    t, r_prev = 1.0, np.inf
    resids, objs = [], []
    for _ in range(niters):
        t_new = (1 + np.sqrt(1 + 4*t**2)) / 2
        beta  = (t - 1) / t_new
        y     = z + beta * (z - z_prev)
        x = prox_quad(y, ALPHA); u = prox_box(2*x - y, ALPHA)
        r = np.linalg.norm(u - x)
        if r > r_prev:
            t_new = 1.0
            x_s = prox_quad(z, ALPHA); u_s = prox_box(2*x_s - z, ALPHA)
            r   = np.linalg.norm(u_s - x_s)
            z_prev = z.copy(); z = z + u_s - x_s; u = u_s
        else:
            z_prev = z.copy(); z = y + u - x
        t, r_prev = t_new, r
        resids.append(r); objs.append(objective(u))
    return np.array(resids), np.array(objs)


def run_adrs_m(z0, niters):
    z, z_prev = z0.copy(), z0.copy()
    t = 1.0; resids, objs = [], []; n_in = 0
    for _ in range(niters):
        t_new = (1 + np.sqrt(1 + 4*t**2)) / 2
        beta  = (t - 1) / t_new
        y     = z + beta * (z - z_prev)
        xI = prox_quad(y, ALPHA); uI = prox_box(2*xI - y, ALPHA); rI = np.linalg.norm(uI - xI)
        xS = prox_quad(z, ALPHA); uS = prox_box(2*xS - z, ALPHA); rS = np.linalg.norm(uS - xS)
        if rI <= rS:
            z_prev, z, u, r = z.copy(), y + uI - xI, uI, rI; n_in += 1
        else:
            z_prev, z, u, r = z.copy(), z + uS - xS, uS, rS; t_new = 1.0
        t = t_new
        resids.append(r); objs.append(objective(u))
    print(f"  ADRS-M:    inertial {n_in}/{niters} ({100*n_in/niters:.0f}%)")
    return np.array(resids), np.array(objs)


def run_adrs_m_nm(z0, niters):
    z, z_prev = z0.copy(), z0.copy()
    t = 1.0
    xi = prox_quad(z, ALPHA); ui = prox_box(2*xi - z, ALPHA)
    c  = np.linalg.norm(ui - xi)
    resids, objs = [], []; n_in = 0
    for _ in range(niters):
        t_new = (1 + np.sqrt(1 + 4*t**2)) / 2
        beta  = (t - 1) / t_new
        y     = z + beta * (z - z_prev)
        xI = prox_quad(y, ALPHA); uI = prox_box(2*xI - y, ALPHA); rI = np.linalg.norm(uI - xI)
        if rI <= c:
            z_prev, z, u, r = z.copy(), y + uI - xI, uI, rI; n_in += 1
        else:
            xS = prox_quad(z, ALPHA); uS = prox_box(2*xS - z, ALPHA)
            rS = np.linalg.norm(uS - xS)
            z_prev, z, u, r = z.copy(), z + uS - xS, uS, rS; t_new = 1.0
        c = r; t = t_new
        resids.append(r); objs.append(objective(u))
    print(f"  ADRS-M-NM: inertial {n_in}/{niters} ({100*n_in/niters:.0f}%)")
    return np.array(resids), np.array(objs)


## Run all algorithms

In [ ]:
print("Running all algorithms  (n=500, cond=1e8, 2000 iterations) ...")
r_drs,  o_drs  = run_drs(z0, N_ITER)
r_adrs, o_adrs = run_adrs(z0, N_ITER)
r_m,    o_m    = run_adrs_m(z0, N_ITER)
r_nm,   o_nm   = run_adrs_m_nm(z0, N_ITER)

results = {'DRS': (r_drs, o_drs), 'ADRS': (r_adrs, o_adrs),
           'ADRS-M': (r_m, o_m), 'ADRS-M-NM': (r_nm, o_nm)}

print(f"\nF* = {F_STAR:.4f}")
for name, (r, o) in results.items():
    print(f"  {name:<14} final resid={r[-1]:.2e}   final gap={o[-1]-F_STAR:.2e}")


## Main convergence plots

**Left**: DRS residual $r_k = \|u_k - x_k\|$ — directly comparable to Goldstein et al. Figure 9(a).  
**Right**: Objective suboptimality $F(u_k) - F^\star$.

Goldstein et al. show Fast ADMM+Restart dramatically outperforms vanilla ADMM on this problem. We expect a similar story with ADRS-M vs DRS.


In [ ]:
iters = np.arange(1, N_ITER + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    f"Goldstein et al. QP Benchmark  —  $\\frac{{1}}{{2}}x^\\top Qx + q^\\top x$"
    f"  s.t.  $-\\mathbf{{1}} \\leq x \\leq \\mathbf{{1}}$\n"
    f"n={n}, cond(Q)=10⁸,  {n_active} active constraints at optimum",
    fontsize=12, fontweight='bold')

for name, (r, o) in results.items():
    c, ls = COLORS[name], STYLES[name]
    axes[0].semilogy(iters, r,                          color=c, linestyle=ls, label=name)
    axes[1].semilogy(iters, np.maximum(o - F_STAR, 1e-14), color=c, linestyle=ls, label=name)

axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('$r_k = \\|u_k - x_k\\|$')
axes[0].set_title('DRS residual  (cf. Goldstein et al. Fig. 9a)')
axes[0].legend()

axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('$F(u_k) - F^\\star$')
axes[1].set_title('Objective suboptimality')
axes[1].legend()

plt.tight_layout()
plt.savefig('qp_main.png', dpi=130, bbox_inches='tight')
plt.show()


## Iterations to reach tolerance

In [ ]:
tols_resid = [1e-2, 1e-4, 1e-6, 1e-8]
tols_gap   = [1e+2, 1e+0, 1e-2, 1e-4]

print("Residual convergence:")
print(f"{'Algorithm':<14}", end="")
for t in tols_resid:
    print(f"  r<{t:.0e}", end="")
print()
print("-" * 55)
for name, (r, o) in results.items():
    print(f"{name:<14}", end="")
    for t in tols_resid:
        h = np.where(r < t)[0]
        print(f"  {str(h[0]+1) if len(h) else 'N/A':>8}", end="")
    print()

print("\nObjective gap convergence:")
print(f"{'Algorithm':<14}", end="")
for t in tols_gap:
    print(f"  gap<{t:.0e}", end="")
print()
print("-" * 55)
for name, (r, o) in results.items():
    gaps = o - F_STAR
    print(f"{name:<14}", end="")
    for t in tols_gap:
        h = np.where(gaps < t)[0]
        print(f"  {str(h[0]+1) if len(h) else 'N/A':>9}", end="")
    print()


## Stepsize sensitivity

Goldstein et al. Figure 9(b) shows Fast ADMM+Restart is much less sensitive to $\tau$ (their stepsize) than vanilla ADMM.
We reproduce this for DRS vs ADRS-M across a range of $\alpha$.


In [ ]:
alphas = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]
TOL    = 1e-4   # residual tolerance
N_SWEEP = 1000

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Stepsize sensitivity  (iterations to reach residual < 1e-4)',
             fontweight='bold')

palette = plt.cm.viridis(np.linspace(0.1, 0.9, len(alphas)))
iters_s = np.arange(1, N_SWEEP + 1)

for alpha_s, col in zip(alphas, palette):
    M_s  = Q + (1/alpha_s) * np.eye(n)
    M_si = np.linalg.inv(M_s)

    def pq(z, a, _Mi=M_si, _a=alpha_s): return _Mi @ ((1/_a)*z - q)
    def pb(z, a): return np.clip(z, -1, 1)

    # DRS
    z = z0.copy(); r_d = []
    for _ in range(N_SWEEP):
        x = pq(z, alpha_s); u = pb(2*x-z, alpha_s)
        r_d.append(np.linalg.norm(u - x)); z = z + u - x

    # ADRS-M
    z, zp, t = z0.copy(), z0.copy(), 1.0; r_m_s = []
    for _ in range(N_SWEEP):
        tn = (1+np.sqrt(1+4*t**2))/2; beta = (t-1)/tn; y = z+beta*(z-zp)
        xI = pq(y,alpha_s); uI = pb(2*xI-y,alpha_s); rI = np.linalg.norm(uI-xI)
        xS = pq(z,alpha_s); uS = pb(2*xS-z,alpha_s); rS = np.linalg.norm(uS-xS)
        if rI<=rS: zp,z,r=z.copy(),y+uI-xI,rI
        else:      zp,z,r=z.copy(),z+uS-xS,rS; tn=1.0
        t=tn; r_m_s.append(r)

    axes[0].semilogy(iters_s, r_d,   color=col, label=f'α={alpha_s}')
    axes[1].semilogy(iters_s, r_m_s, color=col, label=f'α={alpha_s}')

for ax, title in zip(axes, ['DRS', 'ADRS-M']):
    ax.axhline(TOL, color='black', lw=1, linestyle='--', label='tol=1e-4')
    ax.set_xlabel('Iteration'); ax.set_ylabel('$r_k$')
    ax.set_title(title); ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('qp_sensitivity.png', dpi=130, bbox_inches='tight')
plt.show()
print("ADRS-M should be consistently faster and less sensitive to alpha than DRS.")


## Comparison with Goldstein et al.

| | Goldstein et al. | Our work |
|---|---|---|
| **Method** | Fast ADMM + restart | ADRS-M (DRS + inertial monitor) |
| **Restart criterion** | Combined residual $c_k < \eta c_{k-1}$ | Monitor: pick inertial or safe by residual |
| **Theory (strongly convex)** | $O(1/k^2)$ dual gap | No rate proven |
| **Theory (weakly convex)** | Convergence without rate | Convergence without rate (same) |
| **Per-iteration cost** | 2 ADMM steps per iter | 4 prox evals (ADRS-M) |
| **Framework** | ADMM with dual variable $\lambda$ | DRS: no dual variable, more general split |

The QP here is equivalent to their setup. If ADRS-M matches or beats their Fast ADMM+Restart on this benchmark, that is a strong empirical result. If it falls short, the gap points to what theory we still need — likely the $\eta$-factor criterion from their Algorithm 8, which we should incorporate into ADRS-M.
